In [ ]:
import re
import pandas as pd
from pathlib import Path
from nltk.tokenize import sent_tokenize, word_tokenize

### 1. Textual Analysis Resources

In [ ]:
# Get the Stop Words shared by https://sraf.nd.edu/textual-analysis/stopwords/
with open('data/StopWords_Generic.txt', 'r') as file:
	Stop_Words = file.read().splitlines()
print(len(Stop_Words))

In [ ]:
# Loughran-McDonald Master Dictionary w/ Sentiment Word Lists
# Available from https://sraf.nd.edu/loughranmcdonald-master-dictionary/
sheet_id = '1y2LVPvRqdggmIhSnHQcEZA5lYbe3vS5w'

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
LM_Dictionary = pd.read_csv(url)

In [ ]:
# List of R&D keywords from Merkley (2014 TAR)
with open('data/R&D_Keywords.txt', 'r') as file:
    RnD_Phrases = file.read().splitlines()
print(len(RnD_Phrases))

### 2. Count keywords

Downlaod the 10-X files from https://sraf.nd.edu/sec-edgar-data/. Example files saved in './data/sec_filings'.

In [ ]:
def count_RnD_sentences(text: str) -> tuple[str, int]:
    """
    Count sentences containing R&D phrases in the given text.
    """

    sentences = sent_tokenize(text) # Tokenize the text into sentences
    count = 0
    RnD_Text = ''
    
    # Check each sentence for the presence of any of the phrases
    for sentence in sentences:
        for phrase in RnD_Phrases:
            if phrase in sentence.lower():
                count += 1
                temp_sentence = re.sub(r'\s+', ' ', 
                                       sentence.replace('\n',' ').strip()
                                       )
                RnD_Text += temp_sentence + '\n'
                break  # If one phrase is found, no need to check others
    if RnD_Text != '':
        RnD_Text = RnD_Text.strip()
    return RnD_Text, count

In [ ]:
all_data = pd.DataFrame(
    columns = ['FileName', 'RnD_Sentences', "RnD_Text"])

In [ ]:
root = Path('data/sec_filing')
File_List = list(root.glob('*_10-K_*.txt'))
File_List

In [ ]:
row=0
for file_path in File_List:
    with open(file_path, 'r') as file:
        text1 = file.read()
    
    Text, Sentence_count = count_RnD_sentences(text1)

    Name = file_path.stem  # Get the file name without directory and extension

    all_data.loc[row] = (Name, Sentence_count, Text)
    row += 1

In [ ]:
print(all_data.loc[all_data['RnD_Sentences'] > 0, 'RnD_Text'])

In [ ]:
all_data['RnD_Text'] = all_data['RnD_Text'].astype(str)
all_data = all_data[all_data['RnD_Text']!='']

all_data[['FILING_DATE', 'temp']] = all_data['FileName'].str.split('_10-K_edgar_data_', expand=True)
all_data[['CIK', 'ACC_NUM']] = all_data['temp'].str.split('_', expand=True)
all_data['ACC_NUM'] = all_data['ACC_NUM'].apply(lambda x: re.sub('.txt', '', x))
all_data['CIK'] = pd.to_numeric(all_data['CIK'], errors='coerce')
all_data['FILING_DATE'] = pd.to_numeric(all_data['FILING_DATE'], errors='coerce')
all_data.dropna(subset=['CIK', 'ACC_NUM','FILING_DATE'],inplace=True)

all_data.drop(columns=['FileName', 'temp'], inplace=True)

### 3. Calculate the readability of R&D Disclosure

In [ ]:
import textstat
all_data['RnD_Fog_Index'] = all_data['RnD_Text'].apply(textstat.gunning_fog)

### 4. Calculate the Sentiment of R&D Disclosure

In [ ]:
# Functions to calculate the Sentiment
Neg_Words = LM_Dictionary[LM_Dictionary['Negative']!=0]['Word'].tolist()
Pos_Words = LM_Dictionary[LM_Dictionary['Positive']!=0]['Word'].tolist()
Uncertain_Words = LM_Dictionary[LM_Dictionary['Uncertainty']!=0]['Word'].tolist()
Complex_Words = LM_Dictionary[LM_Dictionary['Complexity']!=0]['Word'].tolist()

In [ ]:
def Pos_words(text):
    words = word_tokenize(text)
    pos_words = [word for word in words if (word.upper() in Pos_Words)]
    num = len(pos_words)
    return num

def Neg_words(text):
    words = word_tokenize(text)
    neg_words = [word for word in words if (word.upper() in Neg_Words)]
    num = len(neg_words)
    return num

def Complex_words(text):
    words = word_tokenize(text)
    complex_words = [word for word in words if (word.upper() in Complex_Words)]
    num = len(complex_words)
    return num

def Uncertain_words(text):
    words = word_tokenize(text)
    uncertain_words = [word for word in words if (word.upper() in Uncertain_Words)]
    num = len(uncertain_words)
    return num

In [ ]:
all_data['Positive_RnD_Words'] = all_data['RnD_Text'].apply(Pos_words)

In [ ]:
all_data['Negative_RnD_Words'] = all_data['RnD_Text'].apply(Neg_words)

In [ ]:
all_data['Complex_RnD_Words'] = all_data['RnD_Text'].apply(Complex_words)

In [ ]:
all_data['Uncertain_RnD_Words'] = all_data['RnD_Text'].apply(Uncertain_words)

### 5. Save Output for Reuse

We will reuse these R&D-flagged sentences (and their dictionary-based scores) in the next class, where we represent the same text numerically (bag-of-words / TF-IDF) instead of just counting keyword hits.

In [ ]:
all_data.to_csv('data/rnd_disclosures.csv', index=False)